### [Feature Engineering Toolkit](https://medium.com/code-applied/feature-engineering-toolkit-how-feature-engine-supercharges-your-python-code-c9e764b45e88)

In [1]:
!pip install feature_engine --quiet

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 230.0/230.0 kB 2.8 MB/s eta 0:00:00


In [2]:
import numpy as np
import pandas as pd
import seaborn as sns

import warnings
warnings.filterwarnings('ignore')

# Load data
df = sns.load_dataset('attention', index_col=[0])
df.head(3)

,subject,attention,solutions,score
0,1,divided,1,2.0
1,2,divided,1,3.0
2,3,divided,1,3.0


##### **MeanEncoder (Target Encoding)**

In [3]:
from feature_engine.encoding import MeanEncoder

# X and y
X = df.drop('score', axis=1)
y = df['score']

e = MeanEncoder(variables=['attention'])
X_encoded = e.fit_transform(X, y)
X_encoded

,subject,attention,solutions
0,1,5.116667,1
1,2,5.116667,1
2,3,5.116667,1
3,4,5.116667,1
4,5,5.116667,1
5,6,5.116667,1
6,7,5.116667,1
7,8,5.116667,1
8,9,5.116667,1
9,10,5.116667,1


##### **RareLabelEncoder**

In [4]:
from feature_engine.encoding import RareLabelEncoder

# Add line at the end of the df [1, 'not-focused', 0, 0]
df.loc[len(df)] = [1, 'not-focused', 0, 0]
# X and y
X = df.drop('score', axis=1)
y = df['score']

# Rare encoding
r = RareLabelEncoder(tol=0.05, n_categories=2)
X_encoded = r.fit_transform(X)
X_encoded

,subject,attention,solutions
0,1,divided,1
1,2,divided,1
2,3,divided,1
3,4,divided,1
4,5,divided,1
...,...,...,...
56,17,focused,3
57,18,focused,3
58,19,focused,3
59,20,focused,3


##### **OrdinalEncoder**

In [5]:
from feature_engine.encoding import OrdinalEncoder

o = OrdinalEncoder(encoding_method='arbitrary') #you can also use "ordered"
X_encoded = o.fit_transform(X)
X_encoded

,subject,attention,solutions
0,1,0,1
1,2,0,1
2,3,0,1
3,4,0,1
4,5,0,1
...,...,...,...
56,17,1,3
57,18,1,3
58,19,1,3
59,20,1,3


#####  **DecisionTreeEncoder**
*DecisionTreeEncoder* uses decision trees to find smart ways of encoding categories.

In [6]:
from feature_engine.encoding import DecisionTreeEncoder

dt = DecisionTreeEncoder(random_state=42)
X_encoded = dt.fit_transform(X, y)
X_encoded

,subject,attention,solutions
0,1,5.116667,1
1,2,5.116667,1
2,3,5.116667,1
3,4,5.116667,1
4,5,5.116667,1
...,...,...,...
56,17,6.800000,3
57,18,6.800000,3
58,19,6.800000,3
59,20,6.800000,3


##### **MeanMedianImputer**

*MeanMedianImputer* fills numeric gaps using either the mean or median.

In [7]:
from feature_engine.imputation import MeanMedianImputer
import numpy as np

# Add line at the end of the df [1, 'not-focused', NA, 0]
df.loc[len(df)] = [1, 'not-focused', np.nan, 0]
# X and y
X = df.drop('score', axis=1)
y = df['score']

imp = MeanMedianImputer(imputation_method='median')
X_imputed = imp.fit_transform(X)
X_imputed

,subject,attention,solutions
0,1,divided,1.0
1,2,divided,1.0
2,3,divided,1.0
3,4,divided,1.0
4,5,divided,1.0
...,...,...,...
57,18,focused,3.0
58,19,focused,3.0
59,20,focused,3.0
60,1,not-focused,0.0


##### **ArbitraryNumberImputer**


In [8]:
from feature_engine.imputation import ArbitraryNumberImputer

imp = ArbitraryNumberImputer(arbitrary_number=-999)
X_imputed = imp.fit_transform(X)
X_imputed

,subject,attention,solutions
0,1,divided,1.0
1,2,divided,1.0
2,3,divided,1.0
3,4,divided,1.0
4,5,divided,1.0
...,...,...,...
57,18,focused,3.0
58,19,focused,3.0
59,20,focused,3.0
60,1,not-focused,0.0


##### **MissingIndicator**

*MissingIndicator* creates binary columns that signal whether a value was missing.

In [9]:
from feature_engine.imputation import AddMissingIndicator

mi = AddMissingIndicator()
X_with_flags = mi.fit_transform(X)
X_with_flags

,subject,attention,solutions,solutions_na
0,1,divided,1.0,0
1,2,divided,1.0,0
2,3,divided,1.0,0
3,4,divided,1.0,0
4,5,divided,1.0,0
...,...,...,...,...
57,18,focused,3.0,0
58,19,focused,3.0,0
59,20,focused,3.0,0
60,1,not-focused,0.0,0


##### **LogTransformer**

*LogTransformer* applies a logarithm to bring skewed distributions closer to normal.

In [10]:
df.tail(5)

,subject,attention,solutions,score
57,18,focused,3.0,6.0
58,19,focused,3.0,6.0
59,20,focused,3.0,5.0
60,1,not-focused,0.0,0.0
61,1,not-focused,NaN,0.0


In [11]:
# df.drop(df.tail(1).index, inplace=True)
df['solutions'].fillna(df['solutions'].median(), inplace=True)
df['solutions'] = np.where(df['solutions'] == 0.0, df['solutions'].median(), df['solutions'])
df.tail(10)

,subject,attention,solutions,score
52,13,focused,3.0,9.0
53,14,focused,3.0,7.0
54,15,focused,3.0,7.0
55,16,focused,3.0,7.0
56,17,focused,3.0,6.0
57,18,focused,3.0,6.0
58,19,focused,3.0,6.0
59,20,focused,3.0,5.0
60,1,not-focused,2.0,0.0
61,1,not-focused,2.0,0.0


In [12]:
from feature_engine.transformation import LogTransformer

# X and y
X = df.drop('score', axis=1)
y = df['score']

# Log Transformation
lt = LogTransformer(variables=['solutions'])
X_transformed = lt.fit_transform(X)
X_transformed

,subject,attention,solutions
0,1,divided,0.000000
1,2,divided,0.000000
2,3,divided,0.000000
3,4,divided,0.000000
4,5,divided,0.000000
...,...,...,...
57,18,focused,1.098612
58,19,focused,1.098612
59,20,focused,1.098612
60,1,not-focused,0.693147


##### **PowerTransformer**

*PowerTransformer* offers Box-Cox and Yeo-Johnson transformations. It applies power or exponential transformations to numerical variables. When a list of variables is not provided, it transforms all the numerical variables.

In [13]:
from feature_engine.transformation import PowerTransformer

pt = PowerTransformer(variables=['solutions'])
X_transformed = pt.fit_transform(X)
X_transformed

,subject,attention,solutions
0,1,divided,1.000000
1,2,divided,1.000000
2,3,divided,1.000000
3,4,divided,1.000000
4,5,divided,1.000000
...,...,...,...
57,18,focused,1.732051
58,19,focused,1.732051
59,20,focused,1.732051
60,1,not-focused,1.414214


##### **Winsorizer (Outlier Trimmer)**
Outliers can ruin your model’s day. *Winsorizer* caps extreme values to reduce their impact.

In [14]:
from feature_engine.outliers import Winsorizer

# Add line at the end of the df [100, 'winsorize this', 0, 0]
df.loc[len(df)] = [100, 'winsorize this', 0, 0]

# X and y
X = df.drop('score', axis=1)
y = df['score']

# Cap Outliers
w = Winsorizer(capping_method='iqr')
X_winsorized = w.fit_transform(X)
X_winsorized

,subject,attention,solutions
0,1.00,divided,1.0
1,2.00,divided,1.0
2,3.00,divided,1.0
3,4.00,divided,1.0
4,5.00,divided,1.0
...,...,...,...
58,19.00,focused,3.0
59,20.00,focused,3.0
60,1.00,not-focused,2.0
61,1.00,not-focused,2.0


##### **Pipelined Example**

In [15]:
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LinearRegression

# Load Data
df = sns.load_dataset('attention', index_col=[0])
# Add line at the end of the df [1, 'not-focused', NA, 0]
df.loc[len(df)] = [1, 'not-focused', np.nan, 0]

# X and y
X = df.drop('score', axis=1)
y = df['score']

# steps
steps = [
    ('imputer', MeanMedianImputer(imputation_method='mean')),
    ('encoder', MeanEncoder(variables=['attention'])),
    ('model', LinearRegression())
]

# Pipeline
pipe = Pipeline(steps)

# Fit
pipe.fit(X, y)

# Predict
print(f'First 4 predictions: {pipe.predict(X)[:4]}')

# Score
print(f'\nScore: {pipe.score(X, y)}')

First 4 predictions: [4.5222519  4.52137687 4.52050184 4.51962681]

Score: 0.48312043867184395
